In [0]:
%fs ls "abfss://gizmobox@adlsngdatalakehouse.dfs.core.windows.net/landing/operational_data/"

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTs gizmobox_ext
URL 'abfss://gizmobox@adlsngdatalakehouse.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `gizmobox-crd`)
COMMENT 'This External Location is for Gizmobox Project'


In [0]:
%sql
SHOW CATALOGs

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS gizmobox
MANAGED LOCATION 'abfss://gizmobox@adlsngdatalakehouse.dfs.core.windows.net/'
COMMENT 'This catalog is for Gizmobox Project'

In [0]:
%sql
USE CATALOG gizmobox;

CREATE SCHEMA IF NOT EXISTS gizmobox.landing
MANAGED LOCATION 'abfss://gizmobox@adlsngdatalakehouse.dfs.core.windows.net/landing';

In [0]:
%sql
USE CATALOG gizmobox;

CREATE SCHEMA IF NOT EXISTS gizmobox.bronze
MANAGED LOCATION 'abfss://gizmobox@adlsngdatalakehouse.dfs.core.windows.net/bronze';
CREATE SCHEMA IF NOT EXISTS gizmobox.silver
MANAGED LOCATION 'abfss://gizmobox@adlsngdatalakehouse.dfs.core.windows.net/silver';
CREATE SCHEMA IF NOT EXISTS gizmobox.gold
MANAGED LOCATION 'abfss://gizmobox@adlsngdatalakehouse.dfs.core.windows.net/gold';

In [0]:
%sql
SHOW SCHEMAS

In [0]:
%sql
USE CATALOG gizmobox;
USE SCHEMA landing;
CREATE EXTERNAL VOLUME IF NOT EXISTS operational_data
    LOCATION 'abfss://gizmobox@adlsngdatalakehouse.dfs.core.windows.net/landing/operational_data/'

In [0]:
%fs ls '/Volumes/gizmobox/landing/operational_data'

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import to_timestamp

refunds_data = [
    (1, 66, "2025-01-10 11:30:00", 85.75, "Payment Error:Retailer"),
    (2, 69, "2025-01-03 12:40:15", 120.50, "Order Cancelled:Customer"),
    (3, 72, "2025-01-06 14:45:30", 65.00, "Product Returned:Customer"),
    (4, 73, "2025-01-07 16:10:45", 210.99, "Order Cancelled:Customer"),
    (5, 75, "2025-01-09 18:25:00", 45.20, "Payment Error:Retailer"),
    (6, 80, "2025-01-10 09:35:20", 130.15, "Order Cancelled:Customer"),
    (7, 83, "2025-01-12 11:20:40", 150.00, "Product Returned:Customer"),
    (8, 85, "2025-01-14 13:15:30", 89.99, "Order Cancelled:Customer"),
    (9, 89, "2025-01-15 15:00:00", 78.50, "Payment Error:Retailer"),
    (10, 91, "2025-01-17 16:45:15", 250.75, "Product Returned:Customer")
]

schema = StructType([
    StructField("refund_id", IntegerType(), True),
    StructField("payment_id", IntegerType(), True),
    StructField("refund_timestamp", StringType(), True),
    StructField("refund_amount", DoubleType(), True),
    StructField("refund_reason", StringType(), True)
])

refunds_df = spark.createDataFrame(refunds_data, schema)

refunds_df = refunds_df.withColumn(
    "refund_timestamp",
    to_timestamp("refund_timestamp")
)

refunds_df.write.format("delta").saveAsTable("gizmobox.landing.refunds")

In [0]:
%sql
select * from gizmobox.landing.refunds